In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df_o = pd.read_excel('adriel_qualitativas.xlsx')

df = df_o[:]

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('\n', '', regex=True)

print(len(df))

print(df.columns.tolist())

95
['ID', 'Carimbo de data/hora', 'Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro que sou maior de 18 anos e aceito participar por meio do formulário de participação, conforme as disposições da Lei Geral de Proteção de Dados (Lei nº 13.709/2018).Em caso de resposta negativa, o formulário será encerrado, garantindo assim o respeito à autonomia e privacidade dos participantes, conforme estabelecido pela legislação vigente.', 'Qual é a sua idade?', 'Em qual estado do Brasil você mora atualmente?', 'Qual é o seu gênero?', 'Qual é o seu nível de educação formal concluída?', 'Qual seu modelo de trabalho?', 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?', 'Por favor, indique a senioridade da posição que ocupa.', 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?', 'Quantos anos de experiência você tem em funções relacionadas a Segurança de 

In [5]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())


df = df[~(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'] == 'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software')]

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())
print(df.shape)

['Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software'
 'Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
['Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
(83, 103)


In [6]:
df.isna().sum()

ID                                                                                                                                                                                                                                                                                                                                                                                                                                                              0
Carimbo de data/hora                                                                                                                                                                                                                                                                                                                                                                                                                                            0
Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro

In [7]:
df.shape

(83, 103)

Analises

In [8]:
categorias = {
    "3rd Party Vulnerability Scanning": [
        "Dependabot", "PHP Security Checker", "Snyk", "Clair", "Blackduck",
        "Retire.JS", "SRC:CLR", "Anchore", "Node Security Platform", "Trivy",
        "Sonatype", "Whitesource"
    ],
    "Static Application Security Testing": [  # SAST
        "CodeQL", "CheckStyle", "Coverity", "SonarQube", "Bandit", "PHPStan",
        "LGTM", "Brakeman", "ESLint Security", "Dart Analyzer", "DevSkim"
    ],
    "Dynamic Application Security Testing": [  # DAST
        "Qark", "Checkmarx", "CFN NAG"
    ],
    "Secure Configuration/Hardening": [
        "Chef", "Ansible", "Terraform", "Puppet", "Phan", "Seccomp",
        "CloudCustodian", "Turbot"
    ],
    "Compliance/Hardening Checks": [
        "Bane", "Ansible-Lint", "Blackbox", "Puppet-Lint", "InSpec",
        "Kube Audit", "Foodcritic", "Cookstyle", "conftest"
    ],
    "Secrets Management": [
        "Pinterest Knox", "GitLeaks", "CyberArk Conjur",
        "ThoughtWorks Talisman", "Berglas", "Docker Secrets", "Trufflehog"
    ]
}

In [9]:
cols_ferramentas = [
    col for col in df.columns
    if "categorias de ferramentas" in str(col).lower()
]

print(f"\n{len(cols_ferramentas)} colunas encontradas para a pergunta das ferramentas:")
for c in cols_ferramentas:
    print("-", c)

df_ferramentas = df[cols_ferramentas].copy()


6 colunas encontradas para a pergunta das ferramentas:
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [3rd Party Vulnerability Scanning Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Static Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere mar

In [10]:
df_ferramentas = df_ferramentas.applymap(
    lambda x: x.strip() if isinstance(x, str) else x
)


C:\Users\adrie\AppData\Local\Temp\ipykernel_2624\1321544343.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_ferramentas = df_ferramentas.applymap(


In [11]:
df_ferramentas.insert(0, "Respondente", range(1, len(df_ferramentas) + 1))


In [12]:
escala_likert = [
    "Diariamente",
    "Semanalmente",
    "Mensalmente",
    "Anualmente",
    "Não Aplicável",
]

In [13]:
resultado = {}

for categoria in df_ferramentas.columns[1:]:  # ignora a coluna "Respondente"
    resultado[categoria] = {}
    for opcao in escala_likert:
        # Seleciona todas as linhas onde o respondente marcou essa opção
        participantes = df_ferramentas[df_ferramentas[categoria] == opcao][["Respondente", categoria]]

        # Armazena as respostas individuais
        resultado[categoria][opcao] = participantes


In [14]:
exemplo_categoria = list(resultado.keys())[2]
print(f"\nExemplo — Categoria: {exemplo_categoria}")
for opcao, respostas in resultado[exemplo_categoria].items():
    print(f"\nOpção: {opcao}")
    print(respostas)


Exemplo — Categoria: Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]

Opção: Diariamente
    Respondente  \
4             4   
5             5   
24           21   
28           24   
32           28   
33           29   
34           30   
37           33   
54           47   
63           56   
75           66   
77           68   
80           71   
84           75   
91           81   
92           82   

   Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]  
4                                         Diariamente                                          

In [15]:
with pd.ExcelWriter("respostas_individuais_agrupadas.xlsx") as writer:
    for i, (categoria, respostas_por_opcao) in enumerate(resultado.items()):
        frames = []
        for opcao, respostas in respostas_por_opcao.items():
            temp = respostas.copy()
            temp["Opção"] = opcao
            frames.append(temp)
        df_categoria = pd.concat(frames, ignore_index=True)
        nome_aba = f"{categoria[:27]}_{i}"  # garante unicidade e respeita limite de 31 chars
        df_categoria.to_excel(writer, sheet_name=nome_aba, index=False)

print("\nArquivo salvo: respostas_individuais_agrupadas.xlsx")



Arquivo salvo: respostas_individuais_agrupadas.xlsx


In [16]:
#files.download("respostas_individuais_agrupadas.xlsx")

## Criar a planilha

In [17]:
import re

# Definição das respostas diretas, indiretas, grandes e pequenas
resposta_direta = "Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software."
resposta_indireta = "Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa."

resposta_pequena = ["Até 9 funcionários", "De 10 a 49 funcionários", "De 50 a 99 funcionários"]
resposta_grande = ["De 100 a 499 funcionários", "De 500 a 999 funcionários", "Mais de 1000 funcionários"]

resposta_especialistas = ['Penetration Tester', 'Security Engineer', 'Privacy Engineer', 'DevOps Engineer']
respostas_nao_especialistas_list = [
  'Developer (Backend, frontend, fullstack)',
  'Solution Architect',
  'Requirements Engineer',
  'Test Manager / Tester',
  'Product Owner',
  'Project Lead / Project Manager',
  'Data Scientist', 'UX/UI Designer',
]

# Nome das colunas corretas
coluna_experiencia = 'Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'
coluna_tamanhoempresa = 'Qual o tamanho da empresa para a qual você trabalha atualmente?'
coluna_especialistas = 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?'
coluna_sistemas = 'Quais são os tipos de sistemas que você trabalha?'

# Filtra as respostas diretas e indiretas
diretas = df[df[coluna_experiencia] == resposta_direta].copy()
indiretas = df[df[coluna_experiencia] == resposta_indireta].copy()

# Filtra as respostas de empresas grandes e pequenas
grandes = df[df[coluna_tamanhoempresa].isin(resposta_grande)].copy()
pequenas = df[df[coluna_tamanhoempresa].isin(resposta_pequena)].copy()

# Helper function to check if a string contains any of the keywords
def contains_any_keyword(text, keywords):
    if pd.isna(text):
        return False
    text_lower = str(text).lower()
    for keyword in keywords:
        # Use regex with word boundaries to match whole words, case-insensitive
        if re.search(r'\b' + re.escape(keyword.lower()) + r'\b', text_lower):
            return True
    return False

# Filtra as respostas de especialistas
# Um respondente é considerado especialista se sua coluna 'coluna_especialistas'
# contém pelo menos uma das palavras-chave em 'resposta_especialistas'.
is_specialist_series = df[coluna_especialistas].apply(
    lambda x: contains_any_keyword(x, resposta_especialistas)
)
especialistas = df[is_specialist_series].copy()

# Filtra as respostas de não especialistas (aqueles que não são especialistas)
nao_especialistas = df[~is_specialist_series].copy()

In [18]:
print("Quantidade de participantes no grupo Experiência Direta:", len(diretas))
print("Quantidade de participantes no grupo Experiência Indireta:", len(indiretas))
print("--"*35)

print("Quantidade de participantes no grupo Big:", len(grandes))
print("Quantidade de participantes no grupo Small:", len(pequenas))
print("--"*35)

print("Quantidade de participantes no grupo Especialistas:", len(especialistas))
print("Quantidade de participantes no grupo Não Especialistas:", len(nao_especialistas))
print("--"*35)
print(f"Porcentagem de participantes no grupo Big: {len(grandes)/88:.1f}")
print(f"Porcentagem de participantes no grupo Small: {len(pequenas)/88:.1f}")

Quantidade de participantes no grupo Experiência Direta: 62
Quantidade de participantes no grupo Experiência Indireta: 21
----------------------------------------------------------------------
Quantidade de participantes no grupo Big: 65
Quantidade de participantes no grupo Small: 16
----------------------------------------------------------------------
Quantidade de participantes no grupo Especialistas: 62
Quantidade de participantes no grupo Não Especialistas: 21
----------------------------------------------------------------------
Porcentagem de participantes no grupo Big: 0.7
Porcentagem de participantes no grupo Small: 0.2


In [19]:
# Lista de colunas de preocupação
colunas_preocupacao = [
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Revisão de Código]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Modelagem de Ameaças]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Identificação de Casos de Abuso/Mau Uso]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Análise de Riscos Arquiteturais]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Penetração]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Riscos]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Vulnerabilidades]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Segurança Manuais/Automatizados]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Execução de Testes Fuzz]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Engenharia de Requisitos de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Codificação Segura]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Resposta Emergencial ao Código]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Configurações e Gerenciamento de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Monitoramento de Eventos de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Auditorias de Segurança Externas]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Segurança da Cadeia de Suprimentos de Software]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]'
]

preocupacao_mapping = {
    #'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5
}

In [20]:
# Lista para armazenar os resultados
!pip install cliffs-delta
from scipy import stats
from cliffs_delta import cliffs_delta

resultados = []

# Função que calcula as médias, p-value, effect size e retorna os resultados para um par de grupos
def calcular_dados(grupo1, grupo2, coluna):
    grupo1_filtrado = grupo1[grupo1[coluna] != "Não Aplicável"]
    grupo2_filtrado = grupo2[grupo2[coluna] != "Não Aplicável"]
    grupo1_mapeadas = grupo1[coluna].map(preocupacao_mapping)
    grupo2_mapeadas = grupo2[coluna].map(preocupacao_mapping)

    score_grupo1 = grupo1_mapeadas.mean()
    score_grupo2 = grupo2_mapeadas.mean()

    stat, p_value = stats.mannwhitneyu(grupo1_mapeadas.dropna(), grupo2_mapeadas.dropna(), alternative='greater')

    # cliffs delta
    effect_size = cliffs_delta(grupo1_mapeadas, grupo2_mapeadas)

    return score_grupo1, score_grupo2, p_value, effect_size

# Função que calcula a distribuição das respostas na escala Likert e a média ponderada para cada coluna
def calcular_likert_distribuicao(grupo, coluna):
    grupo_filtrado = grupo[grupo[coluna] != "Não Aplicável"]

    grupo_mapeadas = grupo_filtrado[coluna].map(preocupacao_mapping)
    total_respostas = grupo_mapeadas.value_counts().sort_index()
    media_ponderada = sum(total_respostas.index * (total_respostas / total_respostas.sum()))
    return total_respostas.to_dict(), media_ponderada


[notice] A new release of pip is available: 24.1.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'scipy'

In [ ]:
# Loop que percorre cada coluna de preocupação, realiza as análises e armazena os resultados
for coluna in colunas_preocupacao:
    total_respostas, media_ponderada = calcular_likert_distribuicao(df, coluna)

    score_direto, score_indireto, p_value_direto_indireto, effect_size_direto_indireto = calcular_dados(diretas, indiretas, coluna)
    score_grande, score_pequena, p_value_grande_pequena, effect_size_grande_pequena = calcular_dados(grandes, pequenas, coluna)
    score_especialistas, score_nao_especialistas, p_value_especialistas_nao_especialistas, effect_size_especialistas_nao_especialistas = calcular_dados(especialistas, nao_especialistas, coluna)

    # Cria um dicionário com todos os resultados calculados e o adiciona à lista de resultados
    linha = {
        'Statement': coluna,
        'In total': total_respostas,
        'Overall score': media_ponderada,
        'Score Indirect': score_indireto,
        'Score Direct': score_direto,
        'P-value Indirect/Direct': p_value_direto_indireto,
        'Effect Size Indirect/Direct': effect_size_direto_indireto,
        'Score Big': score_grande,
        'Score Small': score_pequena,
        'P-value Big/Small': p_value_grande_pequena,
        'Effect Size Big/Small': effect_size_grande_pequena,
        'Score Specialist': score_especialistas,
        'Score Non-Specialist': score_nao_especialistas,
        'P-value Specialist/Non-Specialist': p_value_especialistas_nao_especialistas,
        'Effect Size Specialist/Non-Specialist': effect_size_especialistas_nao_especialistas,
    }
    resultados.append(linha)

In [ ]:
resultados

[{'Statement': 'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Revisão de Código]',
  'In total': {1: 2, 2: 9, 3: 26, 4: 17, 5: 18},
  'Overall score': 3.5555555555555554,
  'Score Indirect': np.float64(3.4),
  'Score Direct': np.float64(3.5964912280701755),
  'P-value Indirect/Direct': np.float64(0.37264630631684476),
  'Effect Size Indirect/Direct': (-0.22734254992319508, 'small'),
  'Score Big': np.float64(3.618181818181818),
  'Score Small': np.float64(3.533333333333333),
  'P-value Big/Small': np.float64(0.33502499443876116),
  'Effect Size Big/Small': (0.046153846153846156, 'negligible'),
  'Score Specialist': np.float64(3.482142857142857),
  'Score Non-Specialist': np.float64(3.8125),
  'P-value Specialist/Non-Specialist': np.float64(0.8312335438080605),
  'Effect Size Specialist/Non-Specialist': (-0.30721966205837176, 's

In [ ]:
nao_aplicavel_summary = []

for coluna in colunas_preocupacao:
    # Contar as ocorrências de 'Não Aplicável'
    nao_aplicavel_count = df[coluna].value_counts().get('Não Aplicável', 0)

    # Contar o total de respostas válidas (excluindo NaN para o cálculo da porcentagem do que foi respondido)
    total_respostas_validas = df[coluna].dropna().shape[0]

    if total_respostas_validas > 0:
        nao_aplicavel_percentage = (nao_aplicavel_count / total_respostas_validas) * 100
    else:
        nao_aplicavel_percentage = 0

    nao_aplicavel_summary.append({
        'Coluna': coluna,
        'Contagem Não Aplicável': nao_aplicavel_count,
        'Porcentagem Não Aplicável': round(nao_aplicavel_percentage, 2)
    })

df_nao_aplicavel_summary = pd.DataFrame(nao_aplicavel_summary)

print("DataFrame com o resumo de 'Não Aplicável' por coluna:")
display(df_nao_aplicavel_summary)

DataFrame com o resumo de 'Não Aplicável' por coluna:


,Coluna,Contagem Não Aplicável,Porcentagem Não Aplicável
0,"Com base em sua experiência, como você avaliar...",11,13.25
1,"Com base em sua experiência, como você avaliar...",20,24.10
2,"Com base em sua experiência, como você avaliar...",18,21.69
3,"Com base em sua experiência, como você avaliar...",14,16.87
4,"Com base em sua experiência, como você avaliar...",15,18.07
5,"Com base em sua experiência, como você avaliar...",18,21.69
6,"Com base em sua experiência, como você avaliar...",13,15.66
7,"Com base em sua experiência, como você avaliar...",13,15.66
8,"Com base em sua experiência, como você avaliar...",28,33.73
9,"Com base em sua experiência, como você avaliar...",20,24.10


In [ ]:
# Identificar colunas de 'facilidade' e 'efetividade'
colunas_facilidade = [col for col in df_nao_aplicavel_summary['Coluna'] if 'facilidade de uso' in col.lower()]
colunas_efetividade = [col for col in df_nao_aplicavel_summary['Coluna'] if 'efetividade das seguintes práticas' in col.lower()]

# Filtrar o DataFrame de resumo para obter as porcentagens
df_facilidade = df_nao_aplicavel_summary[df_nao_aplicavel_summary['Coluna'].isin(colunas_facilidade)]
df_efetividade = df_nao_aplicavel_summary[df_nao_aplicavel_summary['Coluna'].isin(colunas_efetividade)]

# Calcular a média das porcentagens para cada grupo
media_facilidade = df_facilidade['Porcentagem Não Aplicável'].mean()
media_efetividade = df_efetividade['Porcentagem Não Aplicável'].mean()

# Criar um novo DataFrame com os resultados
df_resumo_aplicavel = pd.DataFrame({
    'Atributo': ['Facilidade de Uso', 'Efetividade'],
    'Média Porcentagem Não Aplicável': [round(media_facilidade, 2), round(media_efetividade, 2)]
})

print("DataFrame com o resumo da média de 'Não Aplicável' por atributo:")
display(df_resumo_aplicavel)

DataFrame com o resumo da média de 'Não Aplicável' por atributo:


,Atributo,Média Porcentagem Não Aplicável
0,Facilidade de Uso,22.74
1,Efetividade,19.35


In [ ]:
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import pandas as pd
import os

# Caminho para salvar o arquivo temporário na pasta 'temp'
temp_dir = 'temp'
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir)
temp_file_path = os.path.join(temp_dir, 'analise_temp.xlsx')

# Carregar o CSV e salvar como arquivo Excel temporário na pasta 'temp'
# df = pd.read_csv('C:/Users/Samsung/OneDrive/Documentos/AISE LAB/TCC_Aryely/Analise_Q1/resultados/analise.csv')
df = pd.DataFrame(resultados)

df.to_excel(temp_file_path, index=False)

# Carregar o arquivo Excel temporário
wb = openpyxl.load_workbook(temp_file_path)
ws = wb.active

# 1. Formatar Cabeçalhos
header_font = Font(bold=True)
header_fill = PatternFill(start_color="CCCCCC", end_color="CCCCCC", fill_type="solid")
header_alignment = Alignment(horizontal="center", vertical="center")
p_value_color = '4EF547'

for cell in ws[1]:  # Linha 1 são os cabeçalhos
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment

# 2. Ajustar Largura das Colunas
for col in ws.columns:
    max_length = 0
    column = col[0].column_letter  # Pegue o nome da coluna
    for cell in col:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = (max_length + 2)
    ws.column_dimensions[column].width = adjusted_width

# 3. Formatar as Células com Números
for row in ws.iter_rows(min_row=2, min_col=2, max_col=ws.max_column):
    for cell in row:
        if isinstance(cell.value, (int, float)):  # Se o valor for numérico
            cell.alignment = Alignment(horizontal="center")
            cell.number_format = '0.00'  # Formato com 2 casas decimais


# 4. Formatação Condicional para P-values
for row in ws.iter_rows(min_row=2, min_col=6, max_col=6):  # Coluna P-value Indirect vs Direct
    for cell in row:
        v = float(cell.value)
        if v < 0.05:
            cell.fill = PatternFill(start_color=p_value_color, end_color=p_value_color, fill_type="solid")

for row in ws.iter_rows(min_row=2, min_col=10, max_col=10):  # Coluna P-value Big vs Small
    for cell in row:
        v = float(cell.value)
        if cell.value < 0.05:
            cell.fill = PatternFill(start_color=p_value_color, end_color=p_value_color, fill_type="solid")

for row in ws.iter_rows(min_row=2, min_col=14, max_col=14):  # Coluna P-value Specialist vs Non-Specialist
    for cell in row:
        v = float(cell.value)
        if cell.value < 0.05:
            cell.fill = PatternFill(start_color=p_value_color, end_color=p_value_color, fill_type="solid")


# Aplicando gradientes no Effect Size
for i, row in enumerate(df.itertuples(), start=2):
    pvalue_indirect_direct = row[6]
    pvalue_big_small = row[10]
    pvalue_specialist_non_specialist = row[14]

    # Get the numerical part of the effect size directly from the DataFrame row
    effect_size_ind_dir_value = row[7][0]
    effect_size_big_small_value = row[11][0]
    effect_size_spec_non_spec_value = row[15][0]

    cell_effect_size_ind_dir = ws.cell(row=i, column=7)
    cell_effect_size_big_small = ws.cell(row=i, column=11)
    cell_effect_size_spec_non_spec = ws.cell(row=i, column=15)

    # Definindo limites para o gradiente
    max_gray = 108  # Cinza mais escuro (cerca de #303030)
    min_gray = 230  # Cinza mais claro (cerca de #E6E6E6)

    # P-value Indirect vs Direct
    if pvalue_indirect_direct < 0.05:
        max_value_abs = df.loc[df["P-value Indirect/Direct"] < 0.05, "Effect Size Indirect/Direct"].apply(lambda x: abs(x[0])).max()
        if max_value_abs > 0:
            normalized_value = abs(effect_size_ind_dir_value) / max_value_abs
        else:
            normalized_value = 0
        gray_value = int(min_gray + (max_gray - min_gray) * normalized_value)
        fill_color = f"{gray_value:02X}{gray_value:02X}{gray_value:02X}"  # Formato hexadecimal RRGGBB
        cell_effect_size_ind_dir.fill = PatternFill(start_color=fill_color, end_color=fill_color, fill_type="solid")

    # P-value Big vs Small
    if pvalue_big_small < 0.05:
        max_value_abs = df.loc[df["P-value Big/Small"] < 0.05, "Effect Size Big/Small"].apply(lambda x: abs(x[0])).max()
        if max_value_abs > 0:
            normalized_value = abs(effect_size_big_small_value) / max_value_abs
        else:
            normalized_value = 0
        gray_value = int(min_gray + (max_gray - min_gray) * normalized_value)
        fill_color = f"{gray_value:02X}{gray_value:02X}{gray_value:02X}"  # Formato hexadecimal RRGGBB
        cell_effect_size_big_small.fill = PatternFill(start_color=fill_color, end_color=fill_color, fill_type="solid")

    # P-value Specialist vs Non-Specialist
    if pvalue_specialist_non_specialist < 0.05:
        max_value_abs = df.loc[df["P-value Specialist/Non-Specialist"] < 0.05, "Effect Size Specialist/Non-Specialist"].apply(lambda x: abs(x[0])).max()
        if max_value_abs > 0:
            normalized_value = abs(effect_size_spec_non_spec_value) / max_value_abs
        else:
            normalized_value = 0
        gray_value = int(min_gray + (max_gray - min_gray) * normalized_value)
        fill_color = f"{gray_value:02X}{gray_value:02X}{gray_value:02X}"  # Formato hexadecimal RRGGBB
        cell_effect_size_spec_non_spec.fill = PatternFill(start_color=fill_color, end_color=fill_color, fill_type="solid")

# 5. Aplicar Bordas
thin_border = Border(left=Side(style='thin'),
                     right=Side(style='thin'),
                     top=Side(style='thin'),
                     bottom=Side(style='thin'))

for row in ws.iter_rows():
    for cell in row:
        cell.border = thin_border

# 6. Congelar a Primeira Linha
ws.freeze_panes = "A2"

# Caminho para salvar o arquivo Excel formatado na pasta 'resultados_excel'
output_dir = 'resultados_excel'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
output_file_path = os.path.join(output_dir, 'analise_formatado_manw_clif.xlsx')
wb.save(output_file_path)

print(f"Arquivo temporário salvo em '{temp_file_path}'")
print(f"Formatação concluída e arquivo salvo como '{output_file_path}'.")

Arquivo temporário salvo em 'temp/analise_temp.xlsx'
Formatação concluída e arquivo salvo como 'resultados_excel/analise_formatado_manw_clif.xlsx'.
